In [1]:
import os
from pathlib import Path
import joblib
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from sklearn.preprocessing import StandardScaler

load_dotenv()

from src.data_fetch import get_data
from src.preprocessing import add_features, split, preprocessing
from src.transformer import StockTransformer
from src.train import train_model
from src.evaluate import evaluate_model

LOOKBACK, HORIZON = 30, 7
EPOCHS, BATCH_SIZE, LEARNING_RATE = 30, 64, 5e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TICKER = os.getenv("STOCK")
PERIOD = f"{os.getenv("PERIOD")}"
INTERVAL = f"{os.getenv("INTERVAL")}"

print("Device:", DEVICE)

Device: cpu


In [2]:
raw_df = get_data(TICKER, PERIOD, INTERVAL)
print(raw_df.shape)
display(raw_df.head())

(3772, 5)


c:\Users\kaush\Files\Codes\Projects\Mini Project\Stock-Forecasting\venv\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
c:\Users\kaush\Files\Codes\Projects\Mini Project\Stock-Forecasting\venv\Lib\site-packages\yfinance\utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
c:\Users\kaush\Files\Codes\Projects\Mini Project\Stock-Forecasting\venv\Lib\site-packages\yfinance\utils.py:587: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pand

,Open,High,Low,Close,Volume
Date,,,,,
2011-11-21,11.070117,11.108373,10.935925,11.028575,447980400
2011-11-22,11.088644,11.295163,11.086253,11.252724,409021200
2011-11-23,11.192952,11.232702,10.964915,10.968203,428271200
2011-11-25,11.010939,11.092530,10.858515,10.865988,254760800
2011-11-28,11.128395,11.259001,11.068024,11.241069,346413200


In [3]:
fig = px.line(
    x=raw_df.index,
    y=raw_df.Close,
    title=f'{TICKER} adjusted closing price',
)

fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Price"
)

fig.show()

## Preprocessing

### Feature Engeinnering

In [4]:
data = add_features(raw_df)
display(data.head())
print("Shape:", data.shape)

,Open,High,Low,Close,Volume,Return,HL_range,OC_change,MA5_ratio,MA20_ratio,Volatility_10,Vol_change
Date,,,,,,,,,,,,
2011-12-19,11.430851,11.501982,11.371375,11.423080,235530400,0.003118,0.011434,-0.000680,0.000063,0.000179,0.008692,-0.581929
2011-12-20,11.588953,11.838210,11.574010,11.833727,337215200,0.035318,0.022326,0.021121,-0.031034,-0.031126,0.014761,0.358881
2011-12-21,11.855842,11.874074,11.715971,11.848670,262948000,0.001262,0.013343,-0.000605,-0.024053,-0.029833,0.014610,-0.248765
2011-12-22,11.865105,11.928765,11.838208,11.911430,202358800,0.005283,0.007603,0.003904,-0.019355,-0.030986,0.014636,-0.261914
2011-12-23,11.945507,12.062066,11.939529,12.054296,269399200,0.011923,0.010165,0.009107,-0.019914,-0.037541,0.014883,0.286152


Shape: (3753, 12)


In [5]:
scalar = StandardScaler()
columns = raw_df.columns
raw_df_scaled = scalar.fit_transform(raw_df)
raw_df = pd.DataFrame(
    columns=columns,
    data=raw_df
)

### Splitting the dataset

In [6]:
train_df, val_df, test_df = split(data, train_size=0.70, val_size=0.10)
print("Train:", train_df.shape, train_df.index.min(), "to", train_df.index.max())
print("Validation:", val_df.shape, val_df.index.min(), "to", val_df.index.max())
print("Test:", test_df.shape, test_df.index.min(), "to", test_df.index.max())

Train: (2627, 12) 2011-12-19 00:00:00 to 2022-05-26 00:00:00
Validation: (375, 12) 2022-05-27 00:00:00 to 2023-11-22 00:00:00
Test: (751, 12) 2023-11-24 00:00:00 to 2026-11-20 00:00:00


In [7]:
((X_train, y_train), (X_val, y_val), (X_test, y_test), x_scaler, y_scaler, feature_names) = preprocessing(
    train_df, val_df, test_df, lookback=LOOKBACK, horizon=HORIZON
)
print("Features:", feature_names)
print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Features: ['Open', 'High', 'Low', 'Close', 'Volume', 'Return', 'HL_range', 'OC_change', 'MA5_ratio', 'MA20_ratio', 'Volatility_10', 'Vol_change']
Train: (2591, 30, 12) (2591, 7)
Validation: (369, 30, 12) (369, 7)
Test: (745, 30, 12) (745, 7)


## Model Training

In [8]:
model = StockTransformer(
    n_features=X_train.shape[-1], horizon=HORIZON, lookback=LOOKBACK,
    d_model=64, nhead=4, num_layers=3,
    dim_feedforward=128, dropout=0.1,
    target_idx=feature_names.index("Close")
)
print(model)

StockTransformer(
  (linear_skip): Linear(in_features=30, out_features=7, bias=True)
  (input_norm): LayerNorm((12,), eps=1e-05, elementwise_affine=True, bias=True)
  (input_dropout): Dropout(p=0.1, inplace=False)
  (input_projection): Linear(in_features=12, out_features=64, bias=True)
  (position): PositionalEncoding()
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-2): 3 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=128, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropo

c:\Users\kaush\Files\Codes\Projects\Mini Project\Stock-Forecasting\ml\src\transformer.py:43: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)


In [9]:
model, history = train_model(
    model, X_train, y_train, X_val, y_val,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE, patience=5, device=DEVICE
)

Epoch 001/30 | train=0.077335 | val=0.022827
Epoch 002/30 | train=0.015096 | val=0.011974
Epoch 003/30 | train=0.009959 | val=0.006391
Epoch 004/30 | train=0.006355 | val=0.008150
Epoch 005/30 | train=0.004283 | val=0.004651
Epoch 006/30 | train=0.003015 | val=0.004059
Epoch 007/30 | train=0.002323 | val=0.003526
Epoch 008/30 | train=0.001817 | val=0.002902
Epoch 009/30 | train=0.001591 | val=0.004180
Epoch 010/30 | train=0.001420 | val=0.002665
Epoch 011/30 | train=0.001225 | val=0.002260
Epoch 012/30 | train=0.001105 | val=0.002758
Epoch 013/30 | train=0.001009 | val=0.002133
Epoch 014/30 | train=0.000932 | val=0.002192
Epoch 015/30 | train=0.000867 | val=0.002575
Epoch 016/30 | train=0.000826 | val=0.002380
Epoch 017/30 | train=0.000775 | val=0.002256
Epoch 018/30 | train=0.000736 | val=0.002570
Early stopping.


## Model Evaluation

In [10]:
fig = go.Figure()
fig.add_trace(go.Scatter(y=history["train_loss"], mode="lines", name="Train"))
fig.add_trace(go.Scatter(y=history["val_loss"], mode="lines", name="Validation"))

fig.update_layout(
    width=900, height=400,
    title="Training history",
    xaxis_title="Epoch",
    yaxis_title="MSE loss",
    template="plotly_white",
)
fig.update_xaxes(showgrid=True, gridcolor="rgba(0,0,0,0.25)")
fig.update_yaxes(showgrid=True, gridcolor="rgba(0,0,0,0.25)")

fig.show()

In [11]:
metrics, actual, predicted = evaluate_model(
    model, X_test, y_test, y_scaler, device=DEVICE
)
display(pd.Series(metrics, name="Test metrics"))

MAE             21.681479
RMSE            30.558141
MAPE_percent     8.617101
Name: Test metrics, dtype: float64

In [12]:
n = min(250, len(actual))

fig = go.Figure()
fig.add_trace(go.Scatter(y=actual[:n], mode="lines", name="Actual"))
fig.add_trace(go.Scatter(y=predicted[:n], mode="lines", name="Predicted"))

fig.update_layout(
    width=1200, height=400,
    title="Test predictions (forecast points pooled across horizons)",
    xaxis_title="Forecast point",
    yaxis_title="Adjusted close",
    template="plotly_white",
)
fig.update_xaxes(showgrid=True, gridcolor="rgba(0,0,0,0.25)")
fig.update_yaxes(showgrid=True, gridcolor="rgba(0,0,0,0.25)")

fig.show()

In [13]:
def forecast_next_sessions(model, feature_df, x_scaler, y_scaler, feature_names, lookback=60, horizon=7, device="cpu"):
    latest = x_scaler.transform(feature_df[feature_names]).astype(np.float32)[-lookback:]
    model.eval()
    with torch.no_grad():
        scaled = model(torch.tensor(latest[None, :, :], dtype=torch.float32).to(device)).cpu().numpy()
    prices = y_scaler.inverse_transform(scaled.reshape(-1, 1)).ravel()
    dates = pd.bdate_range(feature_df.index[-1] + pd.Timedelta(days=1), periods=horizon)
    return pd.DataFrame({"Forecast_Date": dates, "Predicted_Close": prices})

forecast_df = forecast_next_sessions(
    model, data, x_scaler, y_scaler, feature_names, LOOKBACK, HORIZON, DEVICE
)
display(forecast_df)

,Forecast_Date,Predicted_Close
0,2026-11-23,217.123810
1,2026-11-24,357.620056
2,2026-11-25,349.840454
3,2026-11-26,319.573730
4,2026-11-27,302.280914
5,2026-11-30,340.583984
6,2026-12-01,306.835419


## Saving the model for later use

In [14]:
out = Path("outputs")
out.mkdir(exist_ok=True)
ticker = TICKER.replace(".", "_").replace("^", "")

torch.save({
    "model_state_dict": model.cpu().state_dict(),
    "n_features": X_train.shape[-1],
    "horizon": HORIZON,
    "lookback": LOOKBACK,
    "feature_names": feature_names,
    "ticker": TICKER
}, out / f"{ticker}_transformer.pth")

joblib.dump({"x_scaler": x_scaler, "y_scaler": y_scaler}, out / f"{ticker}_scalers.joblib")
raw_df.to_csv(out / f"{ticker}_historical.csv")
forecast_df.to_csv(out / f"{ticker}_forecast.csv", index=False)
print("Saved to:", out.resolve())

Saved to: C:\Users\kaush\Files\Codes\Projects\Mini Project\Stock-Forecasting\ml\outputs
